# PVQD · VoicePrivacy 2026 verifier · original-audio EER
Select a **Colab GPU**. Run the install cell once, **Restart session**, then run all cells from the top.
Downloads, snippets, checkpoints and results stay on Colab. No repository upload is needed.
This uses the actual pretrained WavLM-large + ECAPA attacker, with WavLM frozen during fine-tuning.
Severity balances sampling; the prediction label is **speaker ID**, not severity or diagnosis.
PVQD snippets share a source recording: this is a within-recording baseline, not a cross-visit test.
**No anonymization. Lower EER is better.** This PVQD adaptation is not an official challenge score.


In [ ]:
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
except ImportError as exc:
    raise RuntimeError("Run this notebook on Colab; do not download the assets locally.") from exc
assert Path("/content").is_dir(), "Use a hosted Colab runtime."
PACKAGES = ["torch==2.8.0", "torchaudio==2.8.0", "torchvision==0.23.0", "speechbrain==1.0.3",
            "numpy==2.2.6", "scipy==1.15.3", "pandas==2.2.3", "soundfile==0.13.1",
            "openpyxl==3.1.5", "scikit-learn==1.6.1", "tqdm==4.67.1", "huggingface-hub>=0.28.1,<1.0"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
print("Installed on Colab. Restart the session once, then run all cells from the top.")


## 1. Imports and settings
Default: 3-second snippets, batch size 4, 3 epochs. All settings are here.


In [ ]:
import hashlib
import json
import math
import random
import re
import shutil
import time
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
from scipy.signal import resample_poly
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split
from speechbrain.lobes.features import Fbank
from speechbrain.lobes.models.ECAPA_TDNN import Classifier
from speechbrain.nnet.losses import AdditiveAngularMargin, LogSoftmaxWrapper
from speechbrain.processing.features import InputNormalization
from tqdm.auto import tqdm
from IPython.display import Audio, display

assert "google.colab" in sys.modules and Path("/content").is_dir(), "Run on Colab."
assert torch.cuda.is_available(), "Select Runtime → Change runtime type → GPU."
SEED, SECONDS, BATCH_SIZE, EPOCHS, LR = 2026, 3.0, 4, 3, 1e-4
ROOT = Path("/content/pvqd_voiceprivacy")
DATA, CLIPS, MODELS = ROOT / "datasets/pvqd", ROOT / "snippets", ROOT / "models/voiceprivacy2026"
OUT = ROOT / "lean_results"
for directory in [DATA / "audio", DATA / "metadata", CLIPS, MODELS, OUT]:
    directory.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
rng = np.random.default_rng(SEED)
print(torch.cuda.get_device_name(0), "· results:", OUT)


## 2. Download PVQD and read severity labels
[PVQD v4](https://doi.org/10.17632/9dz247gnyb.4), Patrick R. Walden / The Voice Foundation, CC BY 4.0.
About 539 MB. Downloads are cached on Colab; publisher hashes verify the audio and workbook.
Audio IDs SJ1001/SJ2004 do not match ST1001/ST2004 in the workbook; exclude rather than guess.


In [ ]:
def fetch_json(url):
    request = urllib.request.Request(url, headers={"Accept": "application/vnd.mendeley-public-dataset.1+json"})
    with urllib.request.urlopen(request, timeout=90) as response:
        return json.load(response)

def download(url, path, sha=None):
    assert "google.colab" in sys.modules and Path("/content").is_dir(), "Downloads run only on Colab."
    if path.exists() and (sha is None or hashlib.sha256(path.read_bytes()).hexdigest() == sha):
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    part = path.with_name(path.name + ".part")
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=120) as source, part.open("wb") as target:
                expected_size = int(source.headers.get("Content-Length", 0))
                shutil.copyfileobj(source, target)
            if expected_size and part.stat().st_size != expected_size:
                raise OSError(f"Incomplete download: {path.name}")
            if sha and hashlib.sha256(part.read_bytes()).hexdigest() != sha:
                raise OSError(f"Hash mismatch: {path.name}")
            part.replace(path)
            return
        except OSError:
            part.unlink(missing_ok=True)
            if attempt == 2:
                raise
            time.sleep(2)

api = "https://data.mendeley.com/public-api/datasets/9dz247gnyb"
folders = fetch_json(f"{api}/folders/4")
records = {}
for name in ["Audio Files", "Ratings Spreadsheets"]:
    folder_id = next(f["id"] for f in folders if f["name"] == name)
    records[name] = fetch_json(f"{api}/files?folder_id={folder_id}&version=4&$limit=1000")
audio_records = records["Audio Files"]
assert len(audio_records) == 296, "Unexpected PVQD file listing."
grade_record = next(r for r in records["Ratings Spreadsheets"] if r["filename"] == "grbas_grade_only.xlsx")

def download_pvqd(record):
    info = record["content_details"]
    name = record["filename"]
    assert Path(name).name == name
    directory = DATA / ("audio" if name.lower().endswith(".wav") else "metadata")
    download(info["download_url"], directory / name, info["sha256_hash"])

with ThreadPoolExecutor(max_workers=2) as pool:
    list(tqdm(pool.map(download_pvqd, audio_records + [grade_record]), total=297, desc="PVQD"))
grade = pd.read_excel(DATA / "metadata/grbas_grade_only.xlsx", header=None).iloc[1:]
normalize_id = lambda value: re.sub(r"[^A-Z0-9]", "", str(value).upper())
severity = {normalize_id(row.iloc[0]): str(row.iloc[11]).strip() for _, row in grade.iterrows() if pd.notna(row.iloc[0])}
print("Severity counts in the source annotations:", pd.Series(severity).value_counts().to_dict())


## 3. Make snippets and split speakers
Mono 16 kHz; 3-second windows with a 250 ms gap, up to 6 per speaker. Skip near-silent windows.
Require at least 4 clips: 2 early clips for enrollment and 2 late clips for probes.
Split original speakers 60/20/20, stratified by severity. No speaker or source recording crosses splits.
Fixed windows may contain vowels or sentences. Review examples for residual clinician speech.


In [ ]:
rows, exclusions = [], []
for record in tqdm(audio_records, desc="Snippets"):
    sid = normalize_id(re.sub(r"[_ ]*E[_ ]?NSS$", "", Path(record["filename"]).stem, flags=re.I))
    if sid not in severity:
        exclusions.append({"speaker": sid, "reason": "severity ID mismatch"})
        continue
    audio, sr = sf.read(DATA / "audio" / record["filename"], dtype="float32", always_2d=True)
    audio = audio.mean(axis=1)
    assert np.isfinite(audio).all()
    divisor = math.gcd(sr, 16000)
    audio = resample_poly(audio, 16000 // divisor, sr // divisor).astype("float32")
    length, step = round(SECONDS * 16000), round((SECONDS + .25) * 16000)
    starts = [s for s in range(4000, len(audio) - length - 4000 + 1, step)
              if np.sqrt(np.mean(audio[s:s + length].astype(float) ** 2)) > 1e-3]
    if len(starts) < 4:
        exclusions.append({"speaker": sid, "reason": "fewer than 4 usable clips"})
        continue
    starts = [starts[i] for i in np.linspace(0, len(starts) - 1, min(6, len(starts))).round().astype(int)]
    for position, start in enumerate(starts):
        path = CLIPS / f"{sid}_{start}.wav"
        sf.write(path, audio[start:start + length], 16000, subtype="FLOAT")
        role = "enroll" if position < 2 else "probe" if position >= len(starts) - 2 else "unused"
        rows.append(dict(speaker=sid, severity=severity[sid], path=str(path), start=start, end=start + length,
                         source_hash=record["content_details"]["sha256_hash"], role=role))
clips = pd.DataFrame(rows)
people = clips[["speaker", "severity"]].drop_duplicates()
assert not people.speaker.duplicated().any()
train_people, heldout = train_test_split(people, test_size=.4, stratify=people.severity, random_state=SEED)
dev_people, test_people = train_test_split(heldout, test_size=.5, stratify=heldout.severity, random_state=SEED)
split = {sid: name for name, frame in [("train", train_people), ("dev", dev_people), ("test", test_people)] for sid in frame.speaker}
clips["split"] = clips.speaker.map(split)
assert clips.groupby("speaker").split.nunique().max() == 1
assert clips.groupby("source_hash").split.nunique().max() == 1
for _, group in clips.groupby("speaker"):
    ordered = group.sort_values("start")
    assert np.all(ordered.start.to_numpy()[1:] >= ordered.end.to_numpy()[:-1])
clips.to_csv(OUT / "snippets.csv", index=False)
pd.DataFrame(exclusions).to_csv(OUT / "exclusions.csv", index=False)
display(clips.drop_duplicates("speaker").groupby(["split", "severity"]).size().unstack(fill_value=0))
display(pd.DataFrame(exclusions))


In [ ]:
for row in clips[clips.split == "train"].groupby("severity").head(1).itertuples():
    print(row.speaker, row.severity)
    display(Audio(filename=row.path))


## 4. Download and load the pretrained attacker
About 925 MB compressed. The [official source](https://github.com/Voice-Privacy-Challenge/Voice-Privacy-Challenge-2026)
is pinned to a commit; its license is retained. Only the two needed weight files are extracted.


In [ ]:
REV = "d5c7f4393db3e09247f58a56ca4d340fcf1ed850"
base = f"https://raw.githubusercontent.com/Voice-Privacy-Challenge/Voice-Privacy-Challenge-2026/{REV}"
for relative in ["evaluation/privacy/asv/WavLM.py", "evaluation/privacy/asv/modules.py",
                 "evaluation/privacy/asv/ecapa_model_wavlm_24layer.py", "LICENSE"]:
    download(f"{base}/{relative}", MODELS / "source" / relative)
weight_names = ["WavLM-Large.pt", "embedding_model.ckpt"]
if not all((MODELS / name).exists() for name in weight_names):
    archive = MODELS / "asv_ssl.zip"
    download("https://duke.app.box.com/shared/static/wo07rwi5xcgam1v6qlzfnzb536rqug14", archive)
    with zipfile.ZipFile(archive) as zipped:
        for name in weight_names:
            member = next(n for n in zipped.namelist() if Path(n).name == name)
            with zipped.open(member) as source, (MODELS / name).open("wb") as target:
                shutil.copyfileobj(source, target)
    archive.unlink()
sys.path.insert(0, str(MODELS / "source"))
from evaluation.privacy.asv.WavLM import WavLM, WavLMConfig
from evaluation.privacy.asv.ecapa_model_wavlm_24layer import ECAPA_TDNN_test

def load_embedding(model, path):
    state = torch.load(path, map_location="cpu", weights_only=False)
    state = state.get("state_dict", state)
    model.embedding.load_state_dict({k.removeprefix("module."): v for k, v in state.items()}, strict=True)

class Verifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        checkpoint = torch.load(MODELS / "WavLM-Large.pt", map_location="cpu", weights_only=False)
        self.wavlm = WavLM(WavLMConfig(checkpoint["cfg"]))
        self.wavlm.load_state_dict(checkpoint["model"], strict=True)
        self.wavlm.requires_grad_(False)
        self.embedding = ECAPA_TDNN_test()
        self.fbank = Fbank(n_mels=80)
        self.normalizer = InputNormalization(norm_type="sentence", std_norm=False)
        load_embedding(self, MODELS / "embedding_model.ckpt")

    def train(self, mode=True):
        super().train(mode)
        self.wavlm.eval()
        return self

    def forward(self, wavs):
        fbank = self.normalizer(self.fbank(wavs), torch.ones(len(wavs), device=wavs.device))
        ssl_wavs = F.layer_norm(wavs, (wavs.shape[-1],)) if self.wavlm.cfg.normalize else wavs
        with torch.no_grad():
            features, _ = self.wavlm.extract_features(ssl_wavs, output_layer=self.wavlm.cfg.encoder_layers, ret_layer_results=True)
            layers = torch.stack([x for x, _ in features[1][1:]]).permute(2, 0, 1, 3)
        return self.embedding(fbank, layers).squeeze(1)

model = Verifier().cuda().eval()
print("Loaded official attacker. Trainable embedding parameters:", sum(p.numel() for p in model.embedding.parameters()))


## 5. Embeddings, trials and EER
Normalize clip embeddings, average the two enrollment clips, then compare every probe against every
enrollment identity. EER normalizes target/non-target error rates separately; equal pair counts are unnecessary.


In [ ]:
def waveform_batch(frame):
    return torch.from_numpy(np.stack([sf.read(p, dtype="float32")[0] for p in frame.path])).cuda()

def extract_embeddings(frame):
    model.eval()
    vectors = []
    with torch.inference_mode():
        for start in range(0, len(frame), BATCH_SIZE):
            vectors.append(F.normalize(model(waveform_batch(frame.iloc[start:start + BATCH_SIZE])), dim=-1).cpu().numpy())
    return np.concatenate(vectors)

def score_trials(split_name):
    frame = clips[(clips.split == split_name) & clips.role.isin(["enroll", "probe"])].copy()
    vectors = extract_embeddings(frame)
    enrollment = frame[frame.role == "enroll"]
    identities = enrollment.speaker.unique()
    prototypes = np.stack([vectors[(frame.role == "enroll") & (frame.speaker == sid)].mean(0) for sid in identities])
    prototypes /= np.maximum(np.linalg.norm(prototypes, axis=1, keepdims=True), 1e-12)
    probe = frame[frame.role == "probe"]
    scores = vectors[frame.role == "probe"] @ prototypes.T
    target = probe.speaker.to_numpy()[:, None] == identities[None, :]
    return pd.DataFrame(dict(probe=np.repeat(probe.speaker.to_numpy(), len(identities)),
                             claim=np.tile(identities, len(probe)), target=target.ravel().astype(int), score=scores.ravel()))

def eer(trials):
    far, tpr, _ = roc_curve(trials.target, trials.score, drop_intermediate=False)
    return float(np.interp(0, far + tpr - 1, far))


## 6. Fine-tune and select by development EER
Equal examples per severity category, cycling through speakers within each category.
Smaller categories repeat speakers more often. Loss predicts training-speaker identity.
WavLM stays frozen; train the embedding network and a new speaker classifier. Test speakers are not used here.


In [ ]:
train = clips[clips.split == "train"].reset_index(drop=True)
labels = {sid: i for i, sid in enumerate(sorted(train.speaker.unique()))}
classifier = Classifier(input_size=192, out_neurons=len(labels)).cuda()
loss_function = LogSoftmaxWrapper(AdditiveAngularMargin(margin=.2, scale=30))
parameters = list(model.embedding.parameters()) + list(classifier.parameters())
optimizer = torch.optim.AdamW(parameters, lr=LR, weight_decay=1e-4)
groups = list(train.drop_duplicates("speaker").groupby("severity"))
examples_per_category = max(len(group) for _, group in groups) * 4
best_eer, history = float("inf"), []
for epoch in range(1, EPOCHS + 1):
    indices = []
    for _, group in groups:
        for sid in np.resize(rng.permutation(group.speaker.to_numpy()), examples_per_category):
            indices.append(int(rng.choice(train.index[train.speaker == sid])))
    rng.shuffle(indices)
    model.train()
    classifier.train()
    losses = []
    for start in tqdm(range(0, len(indices), BATCH_SIZE), desc=f"Epoch {epoch}/{EPOCHS}"):
        batch = train.loc[indices[start:start + BATCH_SIZE]]
        if len(batch) < 2:  # ECAPA training BatchNorm needs at least two examples.
            continue
        targets = torch.tensor([labels[sid] for sid in batch.speaker], device="cuda").unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)
        logits = classifier(model(waveform_batch(batch)).unsqueeze(1))
        loss = loss_function(logits, targets)
        assert torch.isfinite(loss), "Non-finite loss. Reduce LR."
        loss.backward()
        torch.nn.utils.clip_grad_norm_(parameters, 5, error_if_nonfinite=True)
        optimizer.step()
        losses.append(loss.item())
    dev_eer = eer(score_trials("dev"))
    history.append(dict(epoch=epoch, loss=float(np.mean(losses)), dev_eer=dev_eer))
    print(history[-1])
    if dev_eer < best_eer:
        best_eer = dev_eer
        torch.save(model.embedding.state_dict(), OUT / "best_finetuned.pt")
pd.DataFrame(history).to_csv(OUT / "training.csv", index=False)


## 7. Compare pretrained and fine-tuned test EER
Both models use the same test speakers, enrollment clips and probes. Save scores and the selected
embedding checkpoint on Colab. Reloading that checkpoint later also needs the frozen WavLM weights.


In [ ]:
results = []
for name, weights in [("pretrained", MODELS / "embedding_model.ckpt"), ("finetuned", OUT / "best_finetuned.pt")]:
    load_embedding(model, weights)
    trials = score_trials("test")
    trials.to_csv(OUT / f"{name}_test_scores.csv", index=False)
    results.append(dict(model=name, EER_percent=100 * eer(trials),
                        target_pairs=int(trials.target.sum()), non_target_pairs=int((1 - trials.target).sum())))
results = pd.DataFrame(results)
results.to_csv(OUT / "eer.csv", index=False)
display(results)
print("Saved on Colab:", OUT)
